# 🗂️ Project 11.4 — 2D Matrix & Advanced Search
**DSA for Mechatronics · Week 11 — Searching Algorithms**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, bisect, time
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A factory monitoring system stores calibration data in sorted 2D matrices
and needs to search them efficiently.

1. **Fully sorted 2D matrix** — rows sorted, first element > last of previous row.
   Treat entire matrix as a 1D sorted array → O(log(m·n)) binary search.
2. **Staircase (row-column sorted) matrix** — each row and column sorted independently.
   Start at top-right: eliminate row or column each step → O(m + n).
3. **Ternary search** on a unimodal sensor response curve — find the peak.
4. **bisect applications** — maintain sorted insert order, count values in range.


## Step 1 — Generate sorted 2D matrices

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
ROWS      = random.randint(4, 6)
COLS      = random.randint(5, 8)
VAL_MAX   = random.randint(200, 400)
N_TARGETS = random.randint(6, 10)
N_CURVE   = random.randint(30, 50)   # sensor curve points

# Matrix 1: fully sorted (row i last < row i+1 first)
total_cells = ROWS * COLS
vals_m1 = sorted(random.sample(range(1, VAL_MAX + 1), total_cells))
MATRIX1 = [vals_m1[r*COLS:(r+1)*COLS] for r in range(ROWS)]

# Matrix 2: row-column sorted (staircase)
# Build by sorting each row, then sort each column — simpler approach:
# fill with random ints, sort rows, then sort columns
import copy
m2 = [[random.randint(1, VAL_MAX) for _ in range(COLS)] for _ in range(ROWS)]
for r in range(ROWS): m2[r].sort()
for c in range(COLS):
    col = sorted(m2[r][c] for r in range(ROWS))
    for r in range(ROWS): m2[r][c] = col[r]
MATRIX2 = m2

# Search targets
all_vals_m1 = [v for row in MATRIX1 for v in row]
present_t = random.sample(all_vals_m1, min(N_TARGETS - 3, len(all_vals_m1)))
absent_t = [random.randint(1, VAL_MAX) for _ in range(3)]
absent_t = [v for v in absent_t if v not in set(all_vals_m1)][:3]
while len(absent_t) < 3:
    v = random.randint(1, VAL_MAX)
    if v not in set(all_vals_m1) and v not in absent_t:
        absent_t.append(v)
TARGETS = present_t + absent_t

print(f"Sorted 2D matrices ({ROWS}×{COLS}):")
print()
print(f"  Matrix 1 (fully sorted — row-major order):")
for row in MATRIX1:
    print(f"    {row}")
print()
print(f"  Matrix 2 (staircase — each row and column sorted):")
for row in MATRIX2:
    print(f"    {row}")
print()
print(f"  Search targets (M1): {TARGETS}")


## Step 2 — Binary search on fully sorted matrix (treat as 1D)

In [ ]:
def search_matrix_1(matrix, target):
    """
    Search in fully sorted matrix (row-major order).

    Key insight: flatten conceptually to 1D sorted array of length m*n.
    Index mapping: 1D index i → row = i // cols,  col = i % cols.

    Standard binary search on virtual [0, m*n - 1] index space.
    O(log(m·n)) = O(log m + log n).
    Never actually flatten the array — just compute row/col on the fly.
    """
    rows, cols = len(matrix), len(matrix[0])
    lo, hi = 0, rows * cols - 1
    steps  = 0
    while lo <= hi:
        steps += 1
        mid = lo + (hi - lo) // 2
        val = matrix[mid // cols][mid % cols]
        if val == target:
            return (mid // cols, mid % cols), steps
        elif val < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return None, steps

print(f"Binary search on fully sorted matrix ({ROWS}×{COLS}):")
print(f"  Total elements: {ROWS*COLS}   log₂({ROWS*COLS}) ≈ {(ROWS*COLS).bit_length()}")
print()
print(f"  {'Target':>8}  {'Found at':>12}  {'Value':>7}  {'Steps':>7}  {'Correct':>8}")
print(f"  {'─'*8}  {'─'*12}  {'─'*7}  {'─'*7}  {'─'*8}")
m1_all_correct = True
for t in TARGETS:
    pos, steps = search_matrix_1(MATRIX1, t)
    if pos:
        r, c = pos
        ok = MATRIX1[r][c] == t
    else:
        ok = t not in all_vals_m1
    if not ok: m1_all_correct = False
    pos_str = f"({pos[0]},{pos[1]})" if pos else "not found"
    val_str = str(MATRIX1[pos[0]][pos[1]]) if pos else "—"
    print(f"  {t:>8}  {pos_str:>12}  {val_str:>7}  {steps:>7}  {'✅' if ok else '❌':>8}")
print()
print(f"  All correct: {'✅ YES' if m1_all_correct else '❌ NO'}")


## Step 3 — Staircase search on row-column sorted matrix

In [ ]:
def search_matrix_staircase(matrix, target):
    """
    Search in row-column sorted matrix (each row and column is sorted).
    This is NOT fully sorted (can't treat as 1D), but we can use the
    staircase algorithm:

    Start at TOP-RIGHT corner (row=0, col=cols-1):
      - If matrix[r][c] == target → found!
      - If matrix[r][c] > target  → eliminate column c (move left, col -= 1)
        because entire column c has values ≥ matrix[r][c] > target.
      - If matrix[r][c] < target  → eliminate row r (move down, row += 1)
        because entire row r has values ≤ matrix[r][c] < target.

    Each step eliminates one row OR one column → O(m + n) total.
    """
    rows, cols = len(matrix), len(matrix[0])
    r, c       = 0, cols - 1
    steps      = 0
    while r < rows and c >= 0:
        steps += 1
        if matrix[r][c] == target:
            return (r, c), steps
        elif matrix[r][c] > target:
            c -= 1                 # eliminate column
        else:
            r += 1                 # eliminate row
    return None, steps

# Use same targets but search in MATRIX2
all_vals_m2 = [v for row in MATRIX2 for v in row]
# pick targets from m2 plus some absent
present_m2 = random.sample(list(set(all_vals_m2)), min(5, len(set(all_vals_m2))))
absent_m2  = [random.randint(1, VAL_MAX) for _ in range(3)]
absent_m2  = [v for v in absent_m2 if v not in set(all_vals_m2)][:3]
while len(absent_m2) < 3:
    v = random.randint(1, VAL_MAX)
    if v not in set(all_vals_m2) and v not in absent_m2:
        absent_m2.append(v)
targets_m2 = present_m2 + absent_m2

print(f"Staircase search on row-column sorted matrix ({ROWS}×{COLS}):")
print(f"  Max steps = {ROWS} + {COLS} - 1 = {ROWS + COLS - 1}  (O(m+n))")
print()
print(f"  {'Target':>8}  {'Found at':>12}  {'Value':>7}  {'Steps':>7}  {'Correct':>8}")
print(f"  {'─'*8}  {'─'*12}  {'─'*7}  {'─'*7}  {'─'*8}")
m2_all_correct = True
for t in targets_m2:
    pos, steps = search_matrix_staircase(MATRIX2, t)
    if pos:
        r, c = pos
        ok = MATRIX2[r][c] == t
    else:
        ok = t not in set(all_vals_m2)
    if not ok: m2_all_correct = False
    pos_str = f"({pos[0]},{pos[1]})" if pos else "not found"
    val_str = str(MATRIX2[pos[0]][pos[1]]) if pos else "—"
    print(f"  {t:>8}  {pos_str:>12}  {val_str:>7}  {steps:>7}  {'✅' if ok else '❌':>8}")
print()
print(f"  All correct: {'✅ YES' if m2_all_correct else '❌ NO'}")


## Step 4 — Ternary search (unimodal sensor curve peak) + bisect

In [ ]:
def ternary_search_peak(arr):
    """
    Find the index of the maximum value in a unimodal array
    (values increase then decrease — like a sensor resonance curve).

    Ternary search: divide into thirds using m1 and m2.
      - If arr[m1] < arr[m2]: peak is in [m1+1, hi] (eliminate left third)
      - Else: peak is in [lo, m2-1] (eliminate right third)
    O(log_{3/2} n) ≈ 2 log₂n / log₂3 comparisons.
    Works for any unimodal function without knowing the peak location.
    """
    lo, hi = 0, len(arr) - 1
    steps  = 0
    while hi - lo > 2:
        steps += 1
        m1 = lo + (hi - lo) // 3
        m2 = hi - (hi - lo) // 3
        if arr[m1] < arr[m2]:
            lo = m1 + 1
        else:
            hi = m2 - 1
    peak_idx = lo + arr[lo:hi+1].index(max(arr[lo:hi+1]))
    return peak_idx, steps

# Generate unimodal sensor response curve
peak_pos  = random.randint(N_CURVE // 4, 3 * N_CURVE // 4)
curve     = []
for i in range(N_CURVE):
    dist = abs(i - peak_pos)
    noise = random.randint(-3, 3)
    val   = max(0, 100 - dist * dist // (N_CURVE // 10) + noise)
    curve.append(val)
# Make strictly unimodal by smoothing slightly
true_peak = curve.index(max(curve))

peak_idx, tern_steps = ternary_search_peak(curve)

print(f"Ternary search — unimodal sensor response curve (n={N_CURVE}):")
print(f"  Curve (first 20): {curve[:20]} ...")
print(f"  True peak index  : {true_peak}  (value={max(curve)})")
print(f"  Ternary search   : index {peak_idx}  (value={curve[peak_idx]})")
print(f"  Correct          : {'✅ YES' if peak_idx == true_peak else '⚠ near peak'}")
print(f"  Steps            : {tern_steps}  (log₂{N_CURVE} ≈ {N_CURVE.bit_length()})")

# bisect applications
print()
N_LOG_TS = random.randint(40, 70)
log_sorted = sorted([random.randint(1, VAL_MAX) for _ in range(N_LOG_TS)])
query_lo   = random.randint(1, VAL_MAX // 2)
query_hi   = random.randint(VAL_MAX // 2, VAL_MAX)
insert_val = random.randint(1, VAL_MAX)

count_in_range = bisect.bisect_right(log_sorted, query_hi) - bisect.bisect_left(log_sorted, query_lo)
insert_pos     = bisect.bisect_left(log_sorted, insert_val)

print(f"bisect module applications (sorted log, n={len(log_sorted)}):")
print(f"  Values in [{query_lo}, {query_hi}]  : {count_in_range} entries  "
      f"(O(log n) range query)")
print(f"  Insert position for {insert_val}    : index {insert_pos}  "
      f"(between {log_sorted[insert_pos-1] if insert_pos>0 else '—'} and "
      f"{log_sorted[insert_pos] if insert_pos<len(log_sorted) else '—'})")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " 2D MATRIX & ADVANCED SEARCH — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  2D MATRIX SEARCH" + " "*(W-18) + "║")
print(f"║  {'Matrix size':<28}: {ROWS}×{COLS} = {ROWS*COLS} elements{'':<11}║")
print(f"║  {'Max binary steps':<28}: {(ROWS*COLS).bit_length()} (log₂{ROWS*COLS}){'':<12}║")
print(f"║  {'M1 (fully sorted) correct':<28}: {'YES ✅' if m1_all_correct else 'NO ❌':<26}║")
print(f"║  {'Staircase max steps':<28}: {ROWS + COLS - 1} (O(m+n)){'':<16}║")
print(f"║  {'M2 (staircase) correct':<28}: {'YES ✅' if m2_all_correct else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  TERNARY SEARCH" + " "*(W-16) + "║")
print(f"║  {'Curve length':<28}: {N_CURVE:<26}║")
print(f"║  {'True peak index':<28}: {true_peak:<26}║")
print(f"║  {'Ternary found index':<28}: {peak_idx:<26}║")
print(f"║  {'Steps':<28}: {tern_steps:<26}║")
print("╠" + "═"*W + "╣")
print("║  BISECT APPLICATIONS" + " "*(W-21) + "║")
print(f"║  {f'Range [{query_lo},{query_hi}]':<28}: {count_in_range} entries{'':<17}║")
print(f"║  {f'Insert pos for {insert_val}':<28}: index {insert_pos:<19}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about searching in this context?

*Your answer here:*

---

**Q2 — Why is binary search O(log n) and not O(n)?**
Explain in your own words why halving the search space each step leads to logarithmic complexity,
and give one example from your results that illustrates this.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
